# Bike-Sharing Data Preprocessing — Exploratory Notebook

This notebook is the exploratory counterpart to the Dagster pipeline under [`src/bike_rental/`](../src/bike_rental/). Each step here corresponds one-to-one with a Dagster asset, so the transformations stay in sync:

| Notebook step | Dagster asset |
| --- | --- |
| Step 2 — Hourly aggregation | [`hourly_rentals`](../src/bike_rental/defs/assets/hourly.py) |
| Step 3 — Time features | [`rentals_with_time_features`](../src/bike_rental/defs/assets/time_features.py) |
| Step 4 — Weather join | [`rentals_with_weather`](../src/bike_rental/defs/assets/weather.py) |
| Step 5 — Holiday flag | [`final_dataset`](../src/bike_rental/defs/assets/final_dataset.py) |

Use the notebook to iterate on logic interactively; once a step is stable, mirror it in the corresponding asset.

# Step 1 — Data Exploration

Importing all the datasets and putting them in pandas.DataFrame. Checking all the labels, datatypes, and making sure values are non-null

In [1]:
import pandas as pd

In [2]:
direct = pd.read_csv(
    "../data/raw/direct_pickup_bike_rentals.csv",
    parse_dates=["datetime"],
    index_col="id",
)
registered = pd.read_csv(
    "../data/raw/registered_bike_rentals.csv",
    parse_dates=["datetime"],
    index_col="id",
)
weather = pd.read_csv(
    "../data/raw/weather.csv", parse_dates=["datetime"], index_col="id"
)
holidays = pd.read_csv(
    "../data/raw/holidays.csv", parse_dates=["date"], index_col="id"
)

In [3]:
direct.head()

,datetime,user_id,location_id
id,,,
1,2011-01-01 00:24:04,232,2
2,2011-01-01 00:30:19,54,14
3,2011-01-01 00:39:08,201,5
4,2011-01-01 01:01:12,298,13
5,2011-01-01 01:02:37,23,14


In [4]:
direct.info()

<class 'pandas.DataFrame'>
RangeIndex: 620017 entries, 1 to 620017
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   datetime     620017 non-null  datetime64[us]
 1   user_id      620017 non-null  int64         
 2   location_id  620017 non-null  int64         
dtypes: datetime64[us](1), int64(2)
memory usage: 14.2 MB


In [5]:
registered.head()

,datetime,user_id,location_id
id,,,
1,2011-01-01 00:05:09,158,16
2,2011-01-01 00:05:21,262,18
3,2011-01-01 00:05:39,68,18
4,2011-01-01 00:12:05,12,9
5,2011-01-01 00:25:58,91,11


In [6]:
registered.info(show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 2672662 entries, 1 to 2672662
Data columns (total 3 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   datetime     2672662 non-null  datetime64[us]
 1   user_id      2672662 non-null  int64         
 2   location_id  2672662 non-null  int64         
dtypes: datetime64[us](1), int64(2)
memory usage: 61.2 MB


In [7]:
weather.head()

,datetime,conditions,temperature_c,perceived_temperature_c,humidity,windspeed_kmh
id,,,,,,
1,2011-01-01 00:00:00,clear,3.3,3.0,81.0,0.0
2,2011-01-01 01:00:00,clear,2.3,2.0,80.0,0.0
3,2011-01-01 02:00:00,clear,2.3,2.0,80.0,0.0
4,2011-01-01 03:00:00,clear,3.3,3.0,75.0,0.0
5,2011-01-01 04:00:00,clear,3.3,3.0,75.0,0.0


In [8]:
weather.info()

<class 'pandas.DataFrame'>
RangeIndex: 17379 entries, 1 to 17379
Data columns (total 6 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   datetime                 17379 non-null  datetime64[us]
 1   conditions               17379 non-null  str           
 2   temperature_c            17379 non-null  float64       
 3   perceived_temperature_c  17379 non-null  float64       
 4   humidity                 17379 non-null  float64       
 5   windspeed_kmh            17379 non-null  float64       
dtypes: datetime64[us](1), float64(4), str(1)
memory usage: 814.8 KB


In [9]:
holidays.head()

,date,holiday
id,,
1,2011-01-17,"Dr. Martin Luther King, Jr.'s Birthday"
2,2011-02-21,Washington's Birthday
3,2011-04-15,D.C. Emancipation Day (observed)
4,2011-05-30,Memorial Day
5,2011-07-04,Independence Day


In [10]:
holidays.info()

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 1 to 21
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   date     21 non-null     datetime64[us]
 1   holiday  21 non-null     str           
dtypes: datetime64[us](1), str(1)
memory usage: 468.0 bytes


# Step 2 — Hourly Aggregation

Aggregate individual rental records to hourly totals for both registered (booked) and direct pickup rentals, then merge into a single hourly dataset.

In [11]:
# Floor each timestamp to the hour boundary
registered["date_hour"] = registered["datetime"].dt.floor("h")
direct["date_hour"] = direct["datetime"].dt.floor("h")

# Count records per hour for each rental type
registered_hourly = (
    registered.groupby("date_hour").size().reset_index(name="registered_count")
)
direct_hourly = (
    direct.groupby("date_hour").size().reset_index(name="direct_count")
)

registered_hourly.rename(columns={"date_hour": "datetime"}, inplace=True)
direct_hourly.rename(columns={"date_hour": "datetime"}, inplace=True)

# Outer merge so no hour is silently dropped if it only appears in one source
hourly_rentals = pd.merge(
    registered_hourly, direct_hourly, on="datetime", how="outer"
).fillna(0)
hourly_rentals["total_count"] = hourly_rentals["registered_count"].astype(
    int
) + hourly_rentals["direct_count"].astype(int)
hourly_rentals = hourly_rentals.sort_values("datetime").reset_index(drop=True)

hourly_rentals.drop(columns=["registered_count", "direct_count"], inplace=True)

print(hourly_rentals.shape)
hourly_rentals.head()

(17379, 2)


,datetime,total_count
0,2011-01-01 00:00:00,16
1,2011-01-01 01:00:00,40
2,2011-01-01 02:00:00,32
3,2011-01-01 03:00:00,13
4,2011-01-01 04:00:00,1


# Step 3 — Time Feature Engineering

Derive temporal features from the timestamp. These give the model signals about daily, weekly, and weekend patterns:

- `hour` — hour of day (0-23)
- `day_of_month` — day of month (1-31)
- `month` — month (1-12)
- `year` — calendar year (2011, 2012)
- `week` — week number (1-52)
- `day_of_week` — day of week (0=Monday … 6=Sunday)
- `is_weekend` — Saturday/Sunday flag
- `date` — calendar date, used later as the join key for the holiday calendar

In [12]:
df = hourly_rentals.copy()

df["hour"] = df["datetime"].dt.hour.astype("int8")  # 0-23
df["day_of_month"] = df["datetime"].dt.day.astype("int8")  # 1-31
df["month"] = df["datetime"].dt.month.astype("int8")  # 1-12
df["year"] = df["datetime"].dt.year.astype("int16")  # e.g. 2018
df["week"] = df["datetime"].dt.isocalendar().week.astype("int8")  # 1-52
df["day_of_week"] = df["datetime"].dt.dayofweek.astype(
    "int8"
)  # 0=Monday, 6=Sunday
df["is_weekend"] = df["day_of_week"] >= 5  # Saturday or Sunday

df["date"] = df["datetime"].dt.date.astype(
    "datetime64[us]"
)  # calendar date, for holiday join

print(df.shape)
df.head()

(17379, 10)


,datetime,total_count,hour,day_of_month,month,year,week,day_of_week,is_weekend,date
0,2011-01-01 00:00:00,16,0,1,1,2011,52,5,True,2011-01-01
1,2011-01-01 01:00:00,40,1,1,1,2011,52,5,True,2011-01-01
2,2011-01-01 02:00:00,32,2,1,1,2011,52,5,True,2011-01-01
3,2011-01-01 03:00:00,13,3,1,1,2011,52,5,True,2011-01-01
4,2011-01-01 04:00:00,1,4,1,1,2011,52,5,True,2011-01-01


# Step 4 — Enrich with Weather Data

Left-join hourly weather onto the rental dataset. Weather is already aligned to exact hour boundaries so no rounding is needed.

In [13]:
df = pd.merge(df, weather, on="datetime", how="left")

print(df.shape)
df.head()

(17379, 15)


,datetime,total_count,hour,day_of_month,month,year,week,day_of_week,is_weekend,date,conditions,temperature_c,perceived_temperature_c,humidity,windspeed_kmh
0,2011-01-01 00:00:00,16,0,1,1,2011,52,5,True,2011-01-01,clear,3.3,3.0,81.0,0.0
1,2011-01-01 01:00:00,40,1,1,1,2011,52,5,True,2011-01-01,clear,2.3,2.0,80.0,0.0
2,2011-01-01 02:00:00,32,2,1,1,2011,52,5,True,2011-01-01,clear,2.3,2.0,80.0,0.0
3,2011-01-01 03:00:00,13,3,1,1,2011,52,5,True,2011-01-01,clear,3.3,3.0,75.0,0.0
4,2011-01-01 04:00:00,1,4,1,1,2011,52,5,True,2011-01-01,clear,3.3,3.0,75.0,0.0


# Step 5 — Add Holiday Information

merge with the holiday calendar with hourly rental dataset, and derive a boolean `is_holiday` flag.

In [14]:
df = pd.merge(df, holidays, on="date", how="left")

# Convert to boolean flag and drop the raw name + helper date column
df["is_holiday"] = df["holiday"].notna()
df = df.drop(columns=["holiday"])
df = df.drop(columns=["date"])

print(df.shape)
df.head()

(17379, 15)


,datetime,total_count,hour,day_of_month,month,year,week,day_of_week,is_weekend,conditions,temperature_c,perceived_temperature_c,humidity,windspeed_kmh,is_holiday
0,2011-01-01 00:00:00,16,0,1,1,2011,52,5,True,clear,3.3,3.0,81.0,0.0,False
1,2011-01-01 01:00:00,40,1,1,1,2011,52,5,True,clear,2.3,2.0,80.0,0.0,False
2,2011-01-01 02:00:00,32,2,1,1,2011,52,5,True,clear,2.3,2.0,80.0,0.0,False
3,2011-01-01 03:00:00,13,3,1,1,2011,52,5,True,clear,3.3,3.0,75.0,0.0,False
4,2011-01-01 04:00:00,1,4,1,1,2011,52,5,True,clear,3.3,3.0,75.0,0.0,False


# Step 6 — Final Verification & Export

Quick sanity checks, then write the prepared dataset to disk.

In [15]:
# Sanity checks
assert df["datetime"].is_monotonic_increasing, "Hours are not sorted"
assert df["datetime"].duplicated().sum() == 0, "Duplicate hours found"
assert df[["total_count"]].ge(0).all().all(), "Negative counts found"

print("All checks passed.")
print(f"\nFinal dataset shape: {df.shape}")
print(f"Date range: {df['datetime'].min()} → {df['datetime'].max()}")
print(f"\nColumn list:\n{df.dtypes}")

df.head(10)

All checks passed.

Final dataset shape: (17379, 15)
Date range: 2011-01-01 00:00:00 → 2012-12-31 23:00:00

Column list:
datetime                   datetime64[us]
total_count                         int64
hour                                 int8
day_of_month                         int8
month                                int8
year                                int16
week                                 int8
day_of_week                          int8
is_weekend                           bool
conditions                            str
temperature_c                     float64
perceived_temperature_c           float64
humidity                          float64
windspeed_kmh                     float64
is_holiday                           bool
dtype: object


,datetime,total_count,hour,day_of_month,month,year,week,day_of_week,is_weekend,conditions,temperature_c,perceived_temperature_c,humidity,windspeed_kmh,is_holiday
0,2011-01-01 00:00:00,16,0,1,1,2011,52,5,True,clear,3.3,3.0,81.0,0.0,False
1,2011-01-01 01:00:00,40,1,1,1,2011,52,5,True,clear,2.3,2.0,80.0,0.0,False
2,2011-01-01 02:00:00,32,2,1,1,2011,52,5,True,clear,2.3,2.0,80.0,0.0,False
3,2011-01-01 03:00:00,13,3,1,1,2011,52,5,True,clear,3.3,3.0,75.0,0.0,False
4,2011-01-01 04:00:00,1,4,1,1,2011,52,5,True,clear,3.3,3.0,75.0,0.0,False
5,2011-01-01 05:00:00,1,5,1,1,2011,52,5,True,clouds,3.3,1.0,75.0,6.0,False
6,2011-01-01 06:00:00,2,6,1,1,2011,52,5,True,clear,2.3,2.0,80.0,0.0,False
7,2011-01-01 07:00:00,3,7,1,1,2011,52,5,True,clear,1.4,1.0,86.0,0.0,False
8,2011-01-01 08:00:00,8,8,1,1,2011,52,5,True,clear,3.3,3.0,75.0,0.0,False
9,2011-01-01 09:00:00,14,9,1,1,2011,52,5,True,clear,7.0,7.0,76.0,0.0,False


In [16]:
from pathlib import Path

output_path = Path("../data/output-local/final_dataset.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)
print(f"Written to {output_path}")

Written to ../data/output-local/final_dataset.csv
